In [18]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Stages 8–9: align all sections
Reads: the VAD sections, their API transcripts, the book.
Writes: data/test/alignment.json — every spoken word, once, with its time in the whole recording, and its score.

After changing anything in `scripts/`, restart the kernel and run all cells.

In [19]:
import sys
import json
from pathlib import Path

sys.path.insert(0, "../../scripts")

from paths import SECTIONS_DIR, SECTIONS_JSON, RAW_RESPONSES, BOOK_TEXT, ALIGNMENT_JSON
from segments import join_segments
from sections import load_sections, to_recording_time, stitch
from uzbek_text import fix_apostrophes, to_model_text
from aligner import load_model, align

In [20]:
model = load_model()

[NeMo I 2026-11-11 19:54:19 mixins:194] Tokenizer SentencePieceTokenizer initialized with 1024 tokens


[NeMo W 2026-11-11 19:54:19 modelPT:175] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /workspace-mnt/final-train-manifests/MCV/train_manifest_final.json
    - /workspace-mnt/final-train-manifests/FLEURS/train_manifest_final.json
    - /workspace-mnt/final-train-manifests/UZBEKVOICE/train_manifest_final.json
    sample_rate: 16000
    batch_size: 32
    shuffle: true
    num_workers: 8
    pin_memory: true
    max_duration: 20
    min_duration: 0.1
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    
[NeMo W 2026-11-11 19:54:19 modelPT:182] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configuration file to setup the

[NeMo I 2026-11-11 19:54:19 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2026-11-11 19:54:19 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-11 19:54:19 label_looping_base:143] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-11-11 19:54:19 rnnt_models:226] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2026-11-11 19:54:19 label_looping_base:143] No conditional node support for Cuda.
    Cuda graphs with while loops are disabled, decoding speed will be slower
    Reason: CUDA is not available


[NeMo I 2026-11-11 19:54:20 save_restore_connector:287] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /home/intern/.cache/huggingface/hub/models--nvidia--stt_uz_fastconformer_hybrid_large_pc/snapshots/cd32f63d620a7b0cb461ed00ecfc45c852eb0bef/stt_uz_fastconformer_hybrid_large_pc.nemo.
[NeMo I 2026-11-11 19:54:20 hybrid_rnnt_ctc_bpe_models:488] No `decoding_cfg` passed when changing decoding strategy, using internal config
[NeMo I 2026-11-11 19:54:20 hybrid_rnnt_ctc_bpe_models:513] Changed decoding strategy of the CTC decoder to 
    strategy: greedy
    preserve_alignments: null
    compute_timestamps: null
    word_seperator: ' '
    segment_seperators:
    - .
    - '!'
    - '?'
    segment_gap_threshold: null
    ctc_timestamp_type: all
    batch_dim_index: 0
    greedy:
      preserve_alignments: false
      compute_timestamps: false
      preserve_frame_confidence: false
      confidence_method_cfg:
        name: entropy
        entropy_type: tsallis
        alp

In [21]:
sections = load_sections(SECTIONS_JSON)
book_words = BOOK_TEXT.read_text(encoding="utf-8").split()

for section in sections:
    name = Path(section["filename"]).stem
    response_path = RAW_RESPONSES / f"{name}.response"
    response = json.loads(response_path.read_text(encoding="utf-8"))

    text = join_segments(response["segments"], book_words)
    section["text"] = to_model_text(fix_apostrophes(text))
    print(f"{name}: {len(section['text'].split())} words")

section_000: 63 words
section_001: 98 words
section_002: 88 words
section_003: 90 words
section_004: 123 words
section_005: 93 words
section_006: 88 words
section_007: 99 words
section_008: 85 words
section_009: 61 words


In [22]:
words_per_section = []

for section in sections:
    audio_path = SECTIONS_DIR / section["filename"]
    words = align(model, audio_path, section["text"])
    words = to_recording_time(words, section)

    for word in words:
        word["section"] = section["filename"]

    words_per_section.append(words)
    print(f"{section['filename']}: aligned {len(words)} words")

all_words = stitch(sections, words_per_section)

print()
print("Total words:", len(all_words))

[NeMo I 2026-11-11 19:54:20 aligner_utils:914] `output_timestep_duration` is not provided, so we calculated that the model output_timestep_duration is 0.08 ms. -- will use this for all batches


[NeMo W 2026-11-11 19:54:20 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-11-11 19:54:20 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:01,  1.11s/it]

section_000.wav: aligned 63 words
[NeMo I 2026-11-11 19:54:21 aligner_utils:914] `output_timestep_duration` is not provided, so we calculated that the model output_timestep_duration is 0.08 ms. -- will use this for all batches



[NeMo W 2026-11-11 19:54:21 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-11-11 19:54:21 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:01,  1.14s/it]


section_001.wav: aligned 98 words
[NeMo I 2026-11-11 19:54:23 aligner_utils:914] `output_timestep_duration` is not provided, so we calculated that the model output_timestep_duration is 0.08 ms. -- will use this for all batches


[NeMo W 2026-11-11 19:54:23 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-11-11 19:54:23 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  1.01it/s]

... (12 outputs left out)

section_008.wav: aligned 85 words
[NeMo I 2026-11-11 19:54:32 aligner_utils:914] `output_timestep_duration` is not provided, so we calculated that the model output_timestep_duration is 0.08 ms. -- will use this for all batches


[NeMo W 2026-11-11 19:54:32 dataloader:881] The following configuration keys are ignored by Lhotse dataloader: use_start_end_token
[NeMo W 2026-11-11 19:54:32 dataloader:533] You are using a non-tarred dataset and requested tokenization during data sampling (pretokenize=True). This will cause the tokenization to happen in the main (GPU) process,possibly impacting the training speed if your tokenizer is very large.If the impact is noticable, set pretokenize=False in dataloader config.(note: that will disable token-per-second filtering and 2D bucketing features)
Transcribing: 1it [00:00,  1.10it/s]


section_009.wav: aligned 61 words


NameError: name 'is_duplicate' is not defined

In [ ]:
out_of_order = 0
for before, after in zip(all_words, all_words[1:]):
    if after["start"] < before["start"]:
        out_of_order += 1
print("Words out of time order:", out_of_order)
print()

for before, after in zip(sections, sections[1:]):
    last_words = [w["word"] for w in all_words if w["section"] == before["filename"]][-4:]
    first_words = [w["word"] for w in all_words if w["section"] == after["filename"]][:4]
    print(f"{before['own_end']:7.2f} s:  ... {' '.join(last_words)} | {' '.join(first_words)} ...")
print()

for word in all_words:
    if word["end"] - word["start"] > 2:
        print("Long word:", word)

Words out of time order: 0

  59.40 s:  ... "Olampanoh, sizga maslahatim bor. | Chinmochin mamlakatida bir daraxt ...
 122.05 s:  ... olib kelmoq kerak ekan. | Oʻsha daraxtdan barg keltirib ...
 178.40 s:  ... joʻnatibdi, Baxtiyor yoʻl yuribdi, | yuribdi, yoʻl yursa ham ...
 238.15 s:  ... kimning jigargoʻshasisan?" deb soʻrabdi. | Men Xuroson podshosining qizi ...
 307.55 s:  ... gazli dudama qilich beribdi. | Yigit Gulnoradan birga ketishni ...
 365.25 s:  ... haybat bilan baqirib yaqinlashibdi. | Baxtiyor ham tavakkal qilib ...
 424.05 s:  ... shahar tinch, osoyishta ekan. | Baxtiyor otda koʻchadan oʻtib ...
 485.60 s:  ... etibsan, balli, debdi. Soʻng | Baxtiyorni daraxti baxtning oldiga ...
 541.75 s:  ... tovoze bilan podshoga beribdi, | Podsho xursand boʻlib, baxtiyorga ...



In [ ]:
print("The 10 words the model was least sure about:")
for word in sorted(all_words, key=lambda w: w["score"])[:10]:
    print(f"   {word['score']:6.2f}  {word['word']:15} {word['start']:7.2f}-{word['end']:7.2f}  {word['section']}")

scores = sorted(word["score"] for word in all_words)
print()
print("Middle score:", scores[len(scores) // 2])

In [ ]:
ALIGNMENT_JSON.write_text(
    json.dumps(all_words, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Saved", len(all_words), "words to", ALIGNMENT_JSON)

Saved 884 words to /home/intern/book_audio_alignment/data/test/alignment.json
